# Gold Layer Tables – Data Engineering Project

This notebook creates analytical Gold Delta tables from the approved `trusted_*` tables produced by the data-quality notebook.

**Source layer:** Trusted / Silver data  
**Target layer:** Gold analytical tables  
**Catalog:** `data_engineering.default`

The Gold tables are designed for reporting and dashboard use while keeping the aggregation logic simple and auditable.


In [0]:
%sql
USE CATALOG data_engineering;
USE SCHEMA default;

SELECT current_catalog() AS active_catalog,
       current_schema() AS active_schema;


active_catalog,active_schema
data_engineering,default


## 1. Gold Table: Daily Transaction Summary

Aggregates trusted transactions by event date.

Metrics:
- Total transactions
- Total reporting amount
- Average reporting amount
- Average risk score
- High-risk transaction count


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_daily_transaction_summary
USING DELTA AS
SELECT
    CAST(DATE(event_timestamp) AS DATE) AS transaction_date,
    COUNT(*) AS total_transactions,
    SUM(COALESCE(amount_reporting_inr, 0)) AS total_amount_reporting_inr,
    AVG(amount_reporting_inr) AS avg_amount_reporting_inr,
    AVG(risk_score) AS avg_risk_score,
    SUM(CASE WHEN risk_score >= 70 THEN 1 ELSE 0 END) AS high_risk_transactions,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_transactions
GROUP BY CAST(DATE(event_timestamp) AS DATE);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM data_engineering.default.gold_daily_transaction_summary
ORDER BY transaction_date
LIMIT 20;


transaction_date,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
2026-01-01,1503,7027592.46,4675.710220,23.494344644045242,5,2026-09-05T09:52:35.625Z
2026-01-02,1520,7096268.74,4668.597855,22.71842105263158,7,2026-09-05T09:52:35.625Z
2026-01-03,1524,6994046.13,4589.269114,23.458661417322833,3,2026-09-05T09:52:35.625Z
2026-01-04,1492,14872104.88,9967.898713,23.029490616621985,5,2026-09-05T09:52:35.625Z
2026-01-05,1501,6905702.19,4600.734304,23.43504330446369,11,2026-09-05T09:52:35.625Z
2026-01-06,1530,7509840.87,4908.392725,22.730718954248367,4,2026-09-05T09:52:35.625Z
2026-01-07,1498,14836681.07,9904.326482,22.48931909212283,5,2026-09-05T09:52:35.625Z
2026-01-08,1526,14902430.74,9765.682005,22.728047182175622,5,2026-09-05T09:52:35.625Z
2026-01-09,1477,7791810.97,5275.430582,23.327014218009477,4,2026-09-05T09:52:35.625Z
2026-01-10,1586,15153187.65,9554.342781,22.798234552332914,8,2026-09-05T09:52:35.625Z


## 2. Gold Table: Merchant Transaction Summary

Combines trusted transactions with trusted merchant master data.

This table supports merchant-level performance and risk analysis.


In [0]:
%sql

CREATE OR REPLACE TABLE data_engineering.default.gold_merchant_summary
USING DELTA AS

SELECT
    t.merchant_id,
    m.merchant_label,
    m.merchant_category,
    m.merchant_country_code,
    m.merchant_risk_tier,
    m.onboarding_date,
    m.merchant_status,
    m.settlement_currency,

    COUNT(*) AS total_transactions,

    SUM(COALESCE(t.amount_reporting_inr, 0))
        AS total_amount_reporting_inr,

    AVG(t.amount_reporting_inr)
        AS avg_amount_reporting_inr,

    AVG(t.risk_score)
        AS avg_risk_score,

    SUM(
        CASE
            WHEN t.risk_score >= 70 THEN 1
            ELSE 0
        END
    ) AS high_risk_transactions,

    current_timestamp() AS gold_created_at

FROM data_engineering.default.trusted_transactions t

LEFT JOIN data_engineering.default.trusted_merchants m
    ON t.merchant_id = m.merchant_id

GROUP BY
    t.merchant_id,
    m.merchant_label,
    m.merchant_category,
    m.merchant_country_code,
    m.merchant_risk_tier,
    m.onboarding_date,
    m.merchant_status,
    m.settlement_currency;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM data_engineering.default.gold_merchant_summary
ORDER BY total_amount_reporting_inr DESC
LIMIT 20;


merchant_id,merchant_label,merchant_category,merchant_country_code,merchant_risk_tier,onboarding_date,merchant_status,settlement_currency,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
MER000092,SYNTH_MERCHANT_00092,ELECTRONICS,IN,LOW,2023-06-30,ACTIVE,INR,105,15834788.68,150807.511238,14.619047619047619,0,2026-09-05T09:52:55.828Z
MER001049,SYNTH_MERCHANT_01049,MARKETPLACE,IN,LOW,2024-10-10,ACTIVE,INR,102,15475481.63,151720.408137,15.0,0,2026-09-05T09:52:55.828Z
MER000638,SYNTH_MERCHANT_00638,MARKETPLACE,IN,LOW,2022-08-14,ACTIVE,INR,117,15474888.46,132264.003932,14.709401709401709,0,2026-09-05T09:52:55.828Z
MER000303,SYNTH_MERCHANT_00303,MARKETPLACE,US,HIGH,2024-01-22,ACTIVE,USD,97,15334402.85,158086.627320,49.27835051546392,10,2026-09-05T09:52:55.828Z
MER001726,SYNTH_MERCHANT_01726,FUEL,IN,LOW,2021-09-14,ACTIVE,INR,91,15294584.59,168072.358132,14.923076923076923,0,2026-09-05T09:52:55.828Z
MER000743,SYNTH_MERCHANT_00743,TRAVEL,IN,MEDIUM,2021-03-31,ACTIVE,INR,117,8943538.01,76440.495812,24.41025641025641,0,2026-09-05T09:52:55.828Z
MER000137,SYNTH_MERCHANT_00137,TRAVEL,IN,MEDIUM,2021-03-13,ACTIVE,INR,93,8835019.83,95000.213226,24.989247311827956,0,2026-09-05T09:52:55.828Z
MER000594,SYNTH_MERCHANT_00594,TRAVEL,IN,LOW,2024-09-12,ACTIVE,INR,114,8763135.45,76869.609211,15.68421052631579,0,2026-09-05T09:52:55.828Z
MER000874,SYNTH_MERCHANT_00874,ELECTRONICS,IN,LOW,2023-02-01,ACTIVE,INR,111,8743028.56,78766.023063,14.018018018018019,0,2026-09-05T09:52:55.828Z
MER000941,SYNTH_MERCHANT_00941,TRAVEL,IN,LOW,2025-10-09,ACTIVE,INR,105,8710786.98,82959.876000,14.771428571428572,0,2026-09-05T09:52:55.828Z


## 3. Gold Table: Customer Transaction Summary

Aggregates trusted transactions for each customer and joins customer master information where available.


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_customer_summary
USING DELTA AS
SELECT
    t.customer_id,
    COUNT(*) AS total_transactions,
    SUM(COALESCE(t.amount_reporting_inr, 0)) AS total_amount_reporting_inr,
    AVG(t.amount_reporting_inr) AS avg_amount_reporting_inr,
    AVG(t.risk_score) AS avg_risk_score,
    SUM(CASE WHEN t.risk_score >= 70 THEN 1 ELSE 0 END) AS high_risk_transactions,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_transactions t
GROUP BY t.customer_id;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM data_engineering.default.gold_customer_summary
ORDER BY total_amount_reporting_inr DESC
LIMIT 20;


customer_id,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
CUS0012507,20,15116196.35,755809.817500,15.0,0,2026-09-05T09:53:14.346Z
CUS0010681,28,7719123.72,275682.990000,19.821428571428573,1,2026-09-05T09:53:14.346Z
CUS0007144,31,7696679.15,248279.972581,18.838709677419356,0,2026-09-05T09:53:14.346Z
CUS0014089,25,7693940.65,307757.626000,9.84,0,2026-09-05T09:53:14.346Z
CUS0016761,30,7692035.03,256401.167667,20.833333333333332,0,2026-09-05T09:53:14.346Z
CUS0009280,26,7682845.75,295494.067308,14.576923076923077,0,2026-09-05T09:53:14.346Z
CUS0008575,31,7677180.51,247650.984194,13.903225806451612,0,2026-09-05T09:53:14.346Z
CUS0014501,14,7674484.72,548177.480000,16.785714285714285,0,2026-09-05T09:53:14.346Z
CUS0000333,22,7673944.47,348815.657727,18.727272727272727,0,2026-09-05T09:53:14.346Z
CUS0000543,33,7670543.17,232440.702121,26.696969696969695,0,2026-09-05T09:53:14.346Z


## 4. Gold Table: Fraud Summary

Summarizes trusted fraud cases and their relationship with trusted transactions.

The table provides:
- Total fraud cases
- Cases linked to transactions
- Average fraud-case closure time in hours when timestamps are available


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_fraud_summary
USING DELTA AS
SELECT
    COUNT(*) AS total_fraud_cases,
    SUM(CASE
        WHEN primary_transaction_id IS NOT NULL THEN 1
        ELSE 0
    END) AS transaction_linked_cases,
    AVG(
        CASE
            WHEN case_closed_timestamp IS NOT NULL
             AND case_created_timestamp IS NOT NULL
            THEN (unix_timestamp(case_closed_timestamp)
                  - unix_timestamp(case_created_timestamp)) / 3600.0
            ELSE NULL
        END
    ) AS avg_case_resolution_hours,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_fraud_cases;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM data_engineering.default.gold_fraud_summary;


total_fraud_cases,transaction_linked_cases,avg_case_resolution_hours,gold_created_at
4117,4117,62.7276484010,2026-09-05T09:53:33.325Z


## 5. Gold Table: Fraud Cases by Day

Provides a daily view of fraud-case volume and linked transaction cases.


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_daily_fraud_summary
USING DELTA AS
SELECT
    CAST(DATE(case_created_timestamp) AS DATE) AS case_date,
    COUNT(*) AS total_fraud_cases,
    SUM(CASE
        WHEN primary_transaction_id IS NOT NULL THEN 1
        ELSE 0
    END) AS linked_transaction_cases,
    AVG(
        CASE
            WHEN case_closed_timestamp IS NOT NULL
             AND case_created_timestamp IS NOT NULL
            THEN (unix_timestamp(case_closed_timestamp)
                  - unix_timestamp(case_created_timestamp)) / 3600.0
            ELSE NULL
        END
    ) AS avg_resolution_hours,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_fraud_cases
GROUP BY CAST(DATE(case_created_timestamp) AS DATE);


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT *
FROM data_engineering.default.gold_daily_fraud_summary
ORDER BY case_date
LIMIT 20;


case_date,total_fraud_cases,linked_transaction_cases,avg_resolution_hours,gold_created_at
2026-01-01,8,8,68.9668056667,2026-09-05T09:53:51.616Z
2026-01-02,25,25,50.4303306667,2026-09-05T09:53:51.616Z
2026-01-03,29,29,50.3395652609,2026-09-05T09:53:51.616Z
2026-01-04,30,30,69.8777778696,2026-09-05T09:53:51.616Z
2026-01-05,32,32,70.8683760385,2026-09-05T09:53:51.616Z
2026-01-06,32,32,70.3186325385,2026-09-05T09:53:51.616Z
2026-01-07,38,38,74.2708332813,2026-09-05T09:53:51.616Z
2026-01-08,31,31,58.2781217619,2026-09-05T09:53:51.616Z
2026-01-09,33,33,57.8753910000,2026-09-05T09:53:51.616Z
2026-01-10,27,27,68.1590403636,2026-09-05T09:53:51.616Z


## 6. Gold Table: Merchant Risk Tier Summary

Summarizes transaction volume, value, and risk by merchant risk tier.


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_merchant_risk_tier_summary
USING DELTA AS
SELECT
    m.merchant_risk_tier,
    COUNT(*) AS total_transactions,
    SUM(COALESCE(t.amount_reporting_inr, 0)) AS total_amount_reporting_inr,
    AVG(t.amount_reporting_inr) AS avg_amount_reporting_inr,
    AVG(t.risk_score) AS avg_risk_score,
    SUM(CASE WHEN t.risk_score >= 70 THEN 1 ELSE 0 END) AS high_risk_transactions,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_transactions t
LEFT JOIN data_engineering.default.trusted_merchants m
    ON t.merchant_id = m.merchant_id
GROUP BY m.merchant_risk_tier;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM data_engineering.default.gold_merchant_risk_tier_summary
ORDER BY avg_risk_score DESC;


merchant_risk_tier,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
HIGH,13974,157801011.72,11292.472572,41.37183340489481,423,2026-09-05T09:54:08.614Z
MEDIUM,65725,598249216.17,9102.308348,27.1559528337771,180,2026-09-05T09:54:08.614Z
LOW,198301,1783283772.22,8992.812806,17.789320275742433,50,2026-09-05T09:54:08.614Z


## 7. Gold Table: Merchant Country Summary

Summarizes transaction activity by merchant country code.


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_merchant_country_summary
USING DELTA AS
SELECT
    m.merchant_country_code,
    COUNT(*) AS total_transactions,
    SUM(COALESCE(t.amount_reporting_inr, 0)) AS total_amount_reporting_inr,
    AVG(t.amount_reporting_inr) AS avg_amount_reporting_inr,
    AVG(t.risk_score) AS avg_risk_score,
    SUM(CASE WHEN t.risk_score >= 70 THEN 1 ELSE 0 END) AS high_risk_transactions,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_transactions t
LEFT JOIN data_engineering.default.trusted_merchants m
    ON t.merchant_id = m.merchant_id
GROUP BY m.merchant_country_code;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM data_engineering.default.gold_merchant_country_summary
ORDER BY total_amount_reporting_inr DESC;


merchant_country_code,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
IN,197836,1899328458.08,9600.519916,18.07535029013931,267,2026-09-05T09:54:33.525Z
US,28067,206754871.94,7366.475645,28.688566644101613,121,2026-09-05T09:54:33.525Z
GB,19472,169339797.99,8696.579601,29.66331142152835,132,2026-09-05T09:54:33.525Z
SG,18590,151547633.06,8152.105060,28.319526627218934,70,2026-09-05T09:54:33.525Z
AE,14035,112363239.04,8005.930819,28.88307801923762,63,2026-09-05T09:54:33.525Z


## 8. Gold Table: Merchant Status Summary

Summarizes transaction performance by merchant status.


In [0]:
%sql
CREATE OR REPLACE TABLE data_engineering.default.gold_merchant_status_summary
USING DELTA AS
SELECT
    m.merchant_status,
    COUNT(*) AS total_transactions,
    SUM(COALESCE(t.amount_reporting_inr, 0)) AS total_amount_reporting_inr,
    AVG(t.amount_reporting_inr) AS avg_amount_reporting_inr,
    AVG(t.risk_score) AS avg_risk_score,
    SUM(CASE WHEN t.risk_score >= 70 THEN 1 ELSE 0 END) AS high_risk_transactions,
    current_timestamp() AS gold_created_at
FROM data_engineering.default.trusted_transactions t
LEFT JOIN data_engineering.default.trusted_merchants m
    ON t.merchant_id = m.merchant_id
GROUP BY m.merchant_status;


num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT * FROM data_engineering.default.gold_merchant_status_summary
ORDER BY total_amount_reporting_inr DESC;


merchant_status,total_transactions,total_amount_reporting_inr,avg_amount_reporting_inr,avg_risk_score,high_risk_transactions,gold_created_at
ACTIVE,270277,2471302439.57,9143.591351,21.151814619815966,627,2026-09-05T09:55:08.910Z
REVIEW,6455,53491135.49,8286.775444,22.451587916343918,20,2026-09-05T09:55:08.910Z
SUSPENDED,1268,14540425.05,11467.212185,22.729495268138802,6,2026-09-05T09:55:08.910Z


## 9. Gold Tables Created

1. `gold_daily_transaction_summary`
2. `gold_merchant_summary`
3. `gold_customer_summary`
4. `gold_fraud_summary`
5. `gold_daily_fraud_summary`
6. `gold_merchant_risk_tier_summary`
7. `gold_merchant_country_summary`
8. `gold_merchant_status_summary`



These tables are intended to be consumed by reporting, dashboard, and analytical workloads.



## 10. Gold Table Validation

Check that the Gold tables were created successfully and display their row counts.


In [0]:
%sql

SELECT 'gold_daily_transaction_summary' AS table_name,
       COUNT(*) AS row_count
FROM data_engineering.default.gold_daily_transaction_summary

UNION ALL

SELECT 'gold_merchant_summary',
       COUNT(*)
FROM data_engineering.default.gold_merchant_summary

UNION ALL

SELECT 'gold_customer_summary',
       COUNT(*)
FROM data_engineering.default.gold_customer_summary

UNION ALL

SELECT 'gold_fraud_summary',
       COUNT(*)
FROM data_engineering.default.gold_fraud_summary

UNION ALL

SELECT 'gold_daily_fraud_summary',
       COUNT(*)
FROM data_engineering.default.gold_daily_fraud_summary

UNION ALL

SELECT 'gold_merchant_risk_tier_summary',
       COUNT(*)
FROM data_engineering.default.gold_merchant_risk_tier_summary

UNION ALL

SELECT 'gold_merchant_country_summary',
       COUNT(*)
FROM data_engineering.default.gold_merchant_country_summary

UNION ALL

SELECT 'gold_merchant_status_summary',
       COUNT(*)
FROM data_engineering.default.gold_merchant_status_summary;

table_name,row_count
gold_daily_transaction_summary,181
gold_merchant_summary,2800
gold_customer_summary,18000
gold_fraud_summary,1
gold_daily_fraud_summary,183
gold_merchant_risk_tier_summary,3
gold_merchant_country_summary,5
gold_merchant_status_summary,3
